In [1]:
from ast import *
from utils import *
import import_ipynb
from x86_ast import *
from rco_test import *
from select_instr import *
from assign_homes import *
from patch_instr import *

In [2]:
def prelude_and_conclusion(p: X86Program) -> X86Program:
        # YOUR CODE HERE
        if p.stack_space % 16 != 0:
                p.stack_space += 8
        prelude = [Instr('pushq',[Reg('rbp')]),
                   Instr('movq',[Reg('rsp'),Reg('rbp')]),
                   Instr('subq',[Immediate(p.stack_space),Reg('rsp')])]
        conclusion = [Instr('addq',[Immediate(p.stack_space),Reg('rsp')]),
                      Instr('popq',[Reg('rbp')]),
                      Instr('retq',[])]
        return X86Program(prelude + p.body + conclusion)
        

In [3]:
if __name__ == "__main__":
    import textwrap
    code = textwrap.dedent("""
    x = 40 + (-9 + 9)
    y = 32
    y = y - x
    print(y + 3)""")
    parsed_code = parse(code)
    rco_code = remove_complex_operands(parsed_code)
    select_instr_code = select_instruction(rco_code)
    assign_homes_code = assign_homes(select_instr_code)
    print(assign_homes_code.stack_space)
    patch_instr_code = patch_instructions(assign_homes_code)
    print(patch_instr_code.stack_space)
    final_program = prelude_and_conclusion(patch_instr_code)
    print(final_program)

40
40
	.globl main
main:
    pushq %rbp
    movq %rsp, %rbp
    subq $48, %rsp
    movq $9, %rax
    negq %rax
    movq %rax, -8(%rbp)
    movq -8(%rbp), %rax
    addq $9, %rax
    movq %rax, -16(%rbp)
    movq $40, %rax
    addq -16(%rbp), %rax
    movq %rax, -24(%rbp)
    movq $32, -32(%rbp)
    movq -24(%rbp), %rax
    subq %rax, -32(%rbp)
    movq -32(%rbp), %rax
    addq $3, %rax
    movq %rax, -40(%rbp)
    movq -40(%rbp), %rdi
    callq print_int
    addq $48, %rsp
    popq %rbp
    retq 


